In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings("ignore")

In [4]:
df = pd.read_csv('../data/processed_data.csv')
print(df.shape)
df.head()

(425, 16)


,Date,Country,Local,Industry Sector,Accident Level,Potential Accident Level,Gender,Employee type,Critical Risk,Description,Year,Month,Day,Weekday,WeekofYear,Season
0,2016-01-01,Country_01,Local_01,Mining,I,IV,Male,Third Party,Pressed,While removing the drill rod of the Jumbo 08 f...,2016,1,1,Friday,53,Summer
1,2016-01-02,Country_02,Local_02,Mining,I,IV,Male,Employee,Pressurized Systems,During the activation of a sodium sulphide pum...,2016,1,2,Saturday,53,Summer
2,2016-01-06,Country_01,Local_03,Mining,I,III,Male,Third Party (Remote),Manual Tools,In the sub-station MILPO located at level +170...,2016,1,6,Wednesday,1,Summer
3,2016-01-08,Country_01,Local_04,Mining,I,I,Male,Third Party,Others,Being 9:45 am. approximately in the Nv. 1880 C...,2016,1,8,Friday,1,Summer
4,2016-01-10,Country_01,Local_04,Mining,IV,IV,Male,Third Party,Others,Approximately at 11:45 a.m. in circumstances t...,2016,1,10,Sunday,1,Summer


In [25]:
# Remove rare classes BEFORE feature creation
class_counts = df[target_col].value_counts()
valid_classes = class_counts[class_counts >= 2].index

df = df[df[target_col].isin(valid_classes)].reset_index(drop=True)

In [26]:
# Target
y = df[target_col]

# Structured
X_struct = pd.get_dummies(df[categorical_cols], drop_first=True)

# Add derived features
X_struct = pd.concat([
    X_struct,
    df[['text_length', 'word_count', 'risk_keyword_score']]
], axis=1)

# Text features
X_tfidf = tfidf.fit_transform(df[text_col])
X_bow = bow.fit_transform(df[text_col])

In [27]:
# Text-based derived features
df['text_length'] = df[text_col].apply(len)
df['word_count'] = df[text_col].apply(lambda x: len(str(x).split()))

# Risk keyword feature
risk_keywords = ['fall', 'failure', 'injury', 'collision', 'leak', 'fire']

def keyword_flag(text):
    text = str(text).lower()
    return sum([1 for word in risk_keywords if word in text])

df['risk_keyword_score'] = df[text_col].apply(keyword_flag)

In [28]:
# Select categorical columns automatically (excluding text + target)
categorical_cols = df.select_dtypes(include='object').columns.tolist()

categorical_cols = [col for col in categorical_cols if col != text_col]

# One-hot encoding
X_struct = pd.get_dummies(df[categorical_cols], drop_first=True)

# Add derived features
X_struct = pd.concat([
    X_struct,
    df[['text_length', 'word_count', 'risk_keyword_score']]
], axis=1)

print("Structured shape:", X_struct.shape)

Structured shape: (424, 351)


In [29]:
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')

X_tfidf = tfidf.fit_transform(df[text_col])

In [30]:
bow = CountVectorizer(max_features=5000, stop_words='english')

X_bow = bow.fit_transform(df[text_col])

In [31]:
# Check class distribution
print(df[target_col].value_counts())

Potential Accident Level
3    143
2    106
1     95
0     49
4     31
Name: count, dtype: int64


In [32]:
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_struct, y, test_size=0.2, random_state=42, stratify=y
)

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_bow, y, test_size=0.2, random_state=42, stratify=y
)

In [33]:
# RQ1: Ensemble Machine Learning Models
models = {
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
    "GradientBoosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
}

In [34]:
def evaluate_model(model, X_train, X_test, y_train, y_test, label):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    print(f"\n--- {label} ---")
    print("Accuracy:", acc)
    print("F1 Score:", f1)
    print(classification_report(y_test, y_pred))
    
    return acc, f1

In [35]:
print("\n### STRUCTURED DATA ###")

for name, model in models.items():
    evaluate_model(model, X_train_s, X_test_s, y_train_s, y_test_s, name)


### STRUCTURED DATA ###

--- RandomForest ---
Accuracy: 0.5764705882352941
F1 Score: 0.5712869620119962
              precision    recall  f1-score   support

           0       0.80      0.80      0.80        10
           1       0.46      0.58      0.51        19
           2       0.39      0.33      0.36        21
           3       0.68      0.72      0.70        29
           4       1.00      0.33      0.50         6

    accuracy                           0.58        85
   macro avg       0.66      0.55      0.57        85
weighted avg       0.59      0.58      0.57        85


--- GradientBoosting ---
Accuracy: 0.5764705882352941
F1 Score: 0.5725268262370029
              precision    recall  f1-score   support

           0       0.89      0.80      0.84        10
           1       0.50      0.32      0.39        19
           2       0.42      0.67      0.52        21
           3       0.68      0.66      0.67        29
           4       0.67      0.33      0.44        

In [36]:
print("\n### TF-IDF ###")

for name, model in models.items():
    evaluate_model(model, X_train_t, X_test_t, y_train_t, y_test_t, name)


### TF-IDF ###

--- RandomForest ---
Accuracy: 0.5058823529411764
F1 Score: 0.48005212211466863
              precision    recall  f1-score   support

           0       1.00      0.60      0.75        10
           1       0.31      0.21      0.25        19
           2       0.43      0.29      0.34        21
           3       0.50      0.86      0.63        29
           4       1.00      0.33      0.50         6

    accuracy                           0.51        85
   macro avg       0.65      0.46      0.50        85
weighted avg       0.53      0.51      0.48        85


--- GradientBoosting ---
Accuracy: 0.4823529411764706
F1 Score: 0.47551162248781415
              precision    recall  f1-score   support

           0       0.86      0.60      0.71        10
           1       0.41      0.37      0.39        19
           2       0.28      0.24      0.26        21
           3       0.53      0.69      0.60        29
           4       0.60      0.50      0.55         6

   

In [37]:
print("\n### BAG OF WORDS ###")

for name, model in models.items():
    evaluate_model(model, X_train_b, X_test_b, y_train_b, y_test_b, name)


### BAG OF WORDS ###

--- RandomForest ---
Accuracy: 0.43529411764705883
F1 Score: 0.4416010325698907
              precision    recall  f1-score   support

           0       1.00      0.70      0.82        10
           1       0.33      0.37      0.35        19
           2       0.19      0.19      0.19        21
           3       0.50      0.59      0.54        29
           4       1.00      0.33      0.50         6

    accuracy                           0.44        85
   macro avg       0.60      0.44      0.48        85
weighted avg       0.48      0.44      0.44        85


--- GradientBoosting ---
Accuracy: 0.4235294117647059
F1 Score: 0.41174373492712596
              precision    recall  f1-score   support

           0       0.86      0.60      0.71        10
           1       0.29      0.21      0.24        19
           2       0.33      0.24      0.28        21
           3       0.40      0.66      0.50        29
           4       1.00      0.33      0.50         

In [39]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [40]:
# Tokenization and padding
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(df[text_col])

X_seq = tokenizer.texts_to_sequences(df[text_col])
X_pad = pad_sequences(X_seq, maxlen=100)

X_train_dl, X_test_dl, y_train_dl, y_test_dl = train_test_split(
    X_pad, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
#Vanilla LSTM
model_lstm = Sequential([
    Embedding(10000, 128, input_length=100),
    LSTM(64),
    Dense(32, activation='relu'),
    Dense(len(np.unique(y)), activation='softmax')
])

model_lstm.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model_lstm.fit(X_train_dl, y_train_dl, epochs=5, batch_size=32, validation_split=0.2)

Epoch 1/5
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.2952 - loss: 1.6010 - val_accuracy: 0.4118 - val_loss: 1.5658
Epoch 2/5
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.3173 - loss: 1.5382 - val_accuracy: 0.4118 - val_loss: 1.4216
Epoch 3/5
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.3173 - loss: 1.4675 - val_accuracy: 0.4265 - val_loss: 1.4430
Epoch 4/5
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.3432 - loss: 1.3917 - val_accuracy: 0.4412 - val_loss: 1.3904
Epoch 5/5
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4502 - loss: 1.2601 - val_accuracy: 0.4559 - val_loss: 1.4921


In [42]:
#Bidirectional LSTM
model_bilstm = Sequential([
    Embedding(10000, 128, input_length=100),
    Bidirectional(LSTM(64)),
    Dense(32, activation='relu'),
    Dense(len(np.unique(y)), activation='softmax')
])

model_bilstm.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model_bilstm.fit(X_train_dl, y_train_dl, epochs=5, batch_size=32, validation_split=0.2)

Epoch 1/5
9/9 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.2915 - loss: 1.5771 - val_accuracy: 0.4118 - val_loss: 1.4857
Epoch 2/5
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.3173 - loss: 1.4878 - val_accuracy: 0.4118 - val_loss: 1.3925
Epoch 3/5
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.3432 - loss: 1.4452 - val_accuracy: 0.4412 - val_loss: 1.3930
Epoch 4/5
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.4649 - loss: 1.3760 - val_accuracy: 0.3529 - val_loss: 1.5029
Epoch 5/5
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5535 - loss: 1.3365 - val_accuracy: 0.4412 - val_loss: 1.3284


In [43]:
#Stacked LSTM
model_stacked = Sequential([
    Embedding(10000, 128, input_length=100),
    LSTM(64, return_sequences=True),
    LSTM(32),
    Dense(32, activation='relu'),
    Dense(len(np.unique(y)), activation='softmax')
])

model_stacked.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model_stacked.fit(X_train_dl, y_train_dl, epochs=5, batch_size=32, validation_split=0.2)

Epoch 1/5
9/9 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - accuracy: 0.2694 - loss: 1.5998 - val_accuracy: 0.4118 - val_loss: 1.5746
Epoch 2/5
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.3173 - loss: 1.5494 - val_accuracy: 0.4118 - val_loss: 1.4776
Epoch 3/5
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.3173 - loss: 1.4776 - val_accuracy: 0.4118 - val_loss: 1.4467
Epoch 4/5
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.3173 - loss: 1.4265 - val_accuracy: 0.4265 - val_loss: 1.4111
Epoch 5/5
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.4428 - loss: 1.3271 - val_accuracy: 0.4412 - val_loss: 1.3456


In [44]:
def evaluate_dl(model, X_test, y_test, name):
    loss, acc = model.evaluate(X_test, y_test)
    print(f"{name} Accuracy:", acc)

evaluate_dl(model_lstm, X_test_dl, y_test_dl, "Vanilla LSTM")
evaluate_dl(model_bilstm, X_test_dl, y_test_dl, "BiLSTM")
evaluate_dl(model_stacked, X_test_dl, y_test_dl, "Stacked LSTM")

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4471 - loss: 1.4032 
Vanilla LSTM Accuracy: 0.4470588266849518
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4353 - loss: 1.4131
BiLSTM Accuracy: 0.43529412150382996
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4235 - loss: 1.3752 
Stacked LSTM Accuracy: 0.42352941632270813
